# Electric Generation Capacity Visualization

This notebook visualizes the available capacity (MW) for grid connection across different substations in Spain. 

**Visual Legend:**
- **Color:** Red (0 MW) $\rightarrow$ Yellow $\rightarrow$ Green (High Capacity).
- **Size:** Radius is proportional to the available capacity.

In [ ]:
import polars as pl
import folium
import branca.colormap as cm
import numpy as np
import os

In [ ]:
data_path = "../data/processed/electric_capacity_merged.parquet"

if not os.path.exists(data_path):
    print(f"Error: Data not found at {data_path}")
else:
    df = pl.read_parquet(data_path)
    
    # Filter valid coordinates
    df_clean = df.filter(
        (pl.col("latitude").is_not_null()) & 
        (pl.col("longitude").is_not_null())
    )
    
    print(f"Loaded {df_clean.height} substations with valid coordinates.")
    display(df_clean.head())

In [ ]:
def create_capacity_map(df):
    # Calculate center
    center_lat = df["latitude"].mean()
    center_lon = df["longitude"].mean()
    
    # Initialize map
    m = folium.Map(location=[center_lat, center_lon], zoom_start=6, tiles="CartoDB Positron")
    
    # Create color map: Red (min) to Green (max)
    max_cap = df["Capacidad disponible (MW)"].max()
    min_cap = df["Capacidad disponible (MW)"].min()
    
    colormap = cm.LinearColormap(
        colors=['#e74c3c', '#f1c40f', '#2ecc71'],
        vmin=0,
        vmax=max_cap if max_cap > 0 else 100,
        caption='Available Capacity (MW)'
    )
    colormap.add_to(m)
    
    # Add circles
    for row in df.to_dicts():
        capacity = row["Capacidad disponible (MW)"]
        
        # Radius scaling (min 3, max 20 based on sqrt for better visual balance)
        # The sqrt ensures high outliers don't take over the map
        radius = 3 + (np.sqrt(capacity) if capacity > 0 else 0) * 1.5
        
        color = colormap(capacity)
        
        popup_text = f"""
        <div style='font-family: sans-serif;'>
            <b>Substation:</b> {row['Subestación']}<br>
            <b>Operator:</b> {row['Gestor de red']}<br>
            <b>Company:</b> {row['company']}<br>
            <b>Available Capacity:</b> <span style='color:{color}'><b>{capacity:.2f} MW</b></span>
        </div>
        """
        
        folium.CircleMarker(
            location=[row['latitude'], row['longitude']],
            radius=radius,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.6,
            popup=folium.Popup(popup_text, max_width=300),
            tooltip=f"{row['Subestación']} ({capacity:.1f} MW)"
        ).add_to(m)
        
    return m

# Generate map
m = create_capacity_map(df_clean)

# Save to HTML for full screen viewing
m.save("electric_capacity_map.html")
print("Map saved to 'electric_capacity_map.html'")

m